# CipherMark — un tatouage cryptographique pour les modèles génératifs**Mémoire de Master · Université Polytechnique de Bamenda**Ce carnet présente l'ensemble du travail expérimental. Toutes les valeurs quiy figurent sont des mesures réelles, produites par les scripts du dépôt etreportées ici telles quelles ; aucune n'est illustrative.---## Le problèmeLes modèles génératifs produisent des images qu'on ne sait plus attribuer.Les tatouages existants souffrent de trois manques :1. **Pas de sécurité cryptographique formelle** — les méthodes répandues   reposent sur des évaluations empiriques.2. **Pas de traçabilité individuelle** — un framework comme DistSeal grave un   message *fixe* dans les poids du décodeur : toutes les générations d'un   même modèle portent le même filigrane.3. **Universalité et sécurité s'excluent** — celui qui couvre plusieurs   architectures renonce aux garanties, celui qui les offre se restreint à la   diffusion.## La propositionUn **champ témoin** Ω, différent à chaque génération :$$\Omega \;=\; \mathrm{HMAC}(K_{\text{secret}},\, h) \;\oplus\; \mathrm{PRG}(s_{\text{master}},\, \text{nonce})$$- `h` est un **hash perceptuel** de l'image → Ω est **lié au contenu** ;- le `nonce` est unique par génération → Ω est **non rejouable** ;- sans les clés, Ω est indistinguable de l'aléatoire → **illisible**.La vérification recalcule ce que Ω devrait valoir et rend un verdict assortid'une p-valeur binomiale.

---# 0 · Comment lire ce carnetLes données sont **embarquées** dans les cellules : le carnet s'exécute telquel, sans accès réseau ni montage de disque. Chaque tableau indique lefichier de résultats dont il provient.La cellule ci-dessous ne fait qu'installer de quoi tracer.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltplt.rcParams.update({    "figure.dpi": 120, "font.size": 9, "axes.grid": True,    "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False,})def tableau(titre, entetes, lignes, largeurs=None):    largeurs = largeurs or [max(len(str(e)), *(len(str(l[i])) for l in lignes)) + 2                            for i, e in enumerate(entetes)]    print(titre); print("=" * sum(largeurs))    print("".join(str(e).ljust(w) for e, w in zip(entetes, largeurs)))    print("-" * sum(largeurs))    for l in lignes:        print("".join(str(c).ljust(w) for c, w in zip(l, largeurs)))    print("=" * sum(largeurs))print("prêt")

---# 1 · La couche cryptographiqueC'est la seule partie du système qui ne coûte ni GPU ni réseau de neurones.Il n'y avait donc aucune raison de la mesurer petit : elle a été poussée à**5 millions de générations pour chacune des six largeurs** de Ω.Le mémoire justifiait le choix de 64 bits par un argument — la sécuritéeffective est bornée par *n*. Le tableau ci-dessous en fait une mesure.*Source : `crypto-six-largeurs-5-millions-de-tirages/`*

In [ ]:
# 5 000 000 de générations par largeur, 8 propriétés vérifiées à chaque fois.CRYPTO = {    #  bits : (collisions, hamming_moyen, avalanche, freq_de_1, monobit_p, n_defauts)    16  : (4934464,   8.00,   7.98, 0.49999, 0.800, 1),    32  : (   2764,  16.00,  16.04, 0.50000, 0.944, 1),    64  : (      0,  32.00,  31.99, 0.49996, 0.111, 0),    96  : (      0,  48.00,  48.00, 0.49996, 0.082, 0),    128 : (      0,  64.00,  64.09, 0.50002, 0.436, 0),    256 : (      0, 128.00, 127.92, 0.50002, 0.177, 0),}tableau("COUCHE CRYPTOGRAPHIQUE — 5 000 000 de tirages par largeur",        ["bits", "collisions", "Hamming moy", "avalanche", "freq de 1", "monobit p", "défauts"],        [[b, f"{v[0]:,}".replace(",", " "), f"{v[1]:.2f}", f"{v[2]:.2f}",          f"{v[3]:.5f}", f"{v[4]:.3f}", v[5]] for b, v in CRYPTO.items()])print()print("Lecture. À 16 bits, 4 934 464 collisions sur 5 000 000 — soit exactement")print("5 000 000 − 65 536 : toutes les valeurs possibles sont épuisées. À 32 bits,")print("2 764 collisions contre 2 911 attendues par le paradoxe des anniversaires.")print("La construction est donc parfaitement aléatoire à toutes les largeurs ;")print("ce qui échoue à 16 et 32 bits, c'est la CAPACITÉ, pas la cryptographie.")print()print("64 bits est la première largeur où les huit propriétés tiennent ensemble.")

In [ ]:
bits = list(CRYPTO)fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.4))a1.semilogy(bits, [max(CRYPTO[b][0], 0.5) for b in bits], "o-", lw=1.8, color="#c0392b")a1.axhline(1, ls="--", lw=1, color="gray")a1.set_xscale("log", base=2); a1.set_xticks(bits); a1.set_xticklabels(bits)a1.set_xlabel("largeur de Ω (bits)"); a1.set_ylabel("collisions sur 5 M")a1.set_title("Unicité du champ témoin")a1.annotate("zéro collision\nà partir de 64 bits", xy=(64, 0.5), xytext=(80, 60),            fontsize=8, arrowprops=dict(arrowstyle="->", lw=0.8))a2.plot(bits, [CRYPTO[b][2] for b in bits], "o-", lw=1.8, label="avalanche mesurée", color="#2c3e50")a2.plot(bits, [b/2 for b in bits], "--", lw=1.2, label="attendu n/2", color="#7f8c8d")a2.set_xscale("log", base=2); a2.set_xticks(bits); a2.set_xticklabels(bits)a2.set_xlabel("largeur de Ω (bits)"); a2.set_ylabel("bits changés")a2.set_title("Avalanche : 1 bit d'entrée modifié"); a2.legend(fontsize=8)plt.tight_layout(); plt.show()

---# 2 · Le hash perceptuelC'est lui qui lie Ω au **contenu** de l'image. Deux exigences contradictoires :il doit rester stable quand l'image subit une compression, et différerfranchement entre deux images distinctes.Mesuré sur **5000 scènes COCO indépendantes** — une image par scène, aucunrecadrage multiple, ce qui fausserait toute statistique de séparation.*Source : `collisions-hash-perceptuel-5000-scenes-coco-independantes.json`*

In [ ]:
HASH = {    "images": 5000, "paires_comparees": 12_497_500, "bits": 256,    "collisions_exactes": 0, "distance_min": 12, "distance_moyenne": 125.23,    "distance_mediane": 126.0, "entropie_effective": 251.9, "bits_morts": 0,    "correlation_moyenne": 0.045,}tableau("HASH PERCEPTUEL — 5000 scènes COCO indépendantes, 12 497 500 paires",        ["propriété", "mesuré", "idéal"],        [["collisions exactes",      HASH["collisions_exactes"],           "0"],         ["distance minimale",       f'{HASH["distance_min"]} bits',       "> dérive légitime"],         ["distance moyenne",        f'{HASH["distance_moyenne"]:.2f} / 256 ({HASH["distance_moyenne"]/256:.1%})', "50 %"],         ["entropie effective",      f'{HASH["entropie_effective"]} / 256 ({HASH["entropie_effective"]/256:.0%})', "256"],         ["bits quasi constants",    HASH["bits_morts"],                   "0"],         ["corrélation moyenne |r|", f'{HASH["correlation_moyenne"]:.3f}', "~0"]])print()print("Une mise en garde méthodologique. Une première mesure donnait 3 collisions")print("et une distance minimale NULLE. C'était un artefact du corpus : les images")print("d'exemple de scikit-image contiennent des doublons — 'chelsea' et 'cat'")print("sont la même photo de chat. Sur COCO pur, zéro collision.")print()print("Compter les collisions sans nommer les paires ne mène nulle part :")print("le script rapporte désormais les vingt plus proches avec leurs noms.")

### La dérive du hash, mesurée en bitsLa vérification compare le hash observé au hash de référence avec un seuil**τ = 69 bits sur 256** (27 %). C'est donc en bits qu'il faut mesurer.*Une métrique en octets aurait été aveugle : dès que 30 % des bits basculent,94 % des octets diffèrent — 0,7⁸ = 5,7 % d'octets intacts — et toutes lesconditions se collent au maximum.**Source : `derive-du-hash-sous-attaques-mesuree-en-bits-5000-images.json`*

In [ ]:
TAU = 69DERIVE = {  # cas : (médiane, p95, max, fraction sous le seuil)    "flou k5":            (26, 43,  75, 0.999),    "JPEG q50":           (29, 47,  88, 0.999),    "redimension ×0,5":   (33, 52,  91, 0.994),    "recadrage 90 %":     (36, 54,  82, 0.999),    "JPEG q30":           (38, 59, 113, 0.990),    "recadrage 50 %":     (69, 96, 126, 0.506),    "IMAGE ÉTRANGÈRE":   (114,132, 146, 0.001),}tableau(f"DÉRIVE DU HASH EN BITS — seuil τ = {TAU}/256, sur 5000 images",        ["condition", "médiane", "p95", "max", "sous le seuil"],        [[k, v[0], v[1], v[2], f"{v[3]:.1%}"] for k, v in DERIVE.items()])print()print("Le chiffre décisif est le dernier : 0,1 % des images ÉTRANGÈRES passent")print("sous le seuil. Ce n'est pas du bruit — c'est le taux d'erreur intrinsèque")print("d'un seuil placé à 27 %, et il prédit exactement le taux de")print("transplantations réussies mesuré indépendamment à la section 4.")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.4))noms = list(DERIVE)[::-1]med  = [DERIVE[n][0] for n in noms]p95  = [DERIVE[n][1] for n in noms]coul = ["#c0392b" if n == "IMAGE ÉTRANGÈRE" else "#7f8c8d" if "50 %" in n else "#2980b9" for n in noms]y = np.arange(len(noms))ax.barh(y, med, color=coul, alpha=0.85, label="médiane")ax.barh(y, np.array(p95) - np.array(med), left=med, color=coul, alpha=0.3, label="jusqu'au p95")ax.axvline(TAU, color="#e67e22", lw=2, ls="--")ax.text(TAU + 2, -0.7, f"seuil τ = {TAU}", color="#e67e22", fontsize=8.5)ax.set_yticks(y); ax.set_yticklabels(noms, fontsize=8.5)ax.set_xlabel("distance de Hamming du hash (bits sur 256)")ax.set_title("Ce que le vérifieur mesure réellement")ax.legend(fontsize=8, loc="lower right")plt.tight_layout(); plt.show()

---# 3 · Robustesse aux transformations passivesDix-sept conditions, **5000 images**, trois modèles comparés. La métrique estle taux de verdicts AUTHENTIC — pas la précision brute, mais ce que levérifieur rend réellement.*Sources : `robustesse-17-conditions-modele-{base-64-bits, phaseB-robuste, 96-bits}-5000-images.json`*

In [ ]:
ROBUSTESSE = {  # condition : (base 64 bits, phase B robuste, 96 bits)    "aucune":        (0.9992, 0.9968, 0.8528),    "jpeg_q90":      (0.9972, 0.9948, 0.8474),    "jpeg_q70":      (0.9918, 0.9912, 0.8430),    "jpeg_q50":      (0.9804, 0.9846, 0.8402),    "jpeg_q30":      (0.9556, 0.9660, 0.8106),    "flou_k3":       (0.9934, 0.9918, 0.7936),    "flou_k5":       (0.9838, 0.9874, 0.5648),    "flou_k7":       (0.7250, 0.9822, 0.1844),    "resize_0.75":   (0.9910, 0.9922, 0.7650),    "resize_0.5":    (0.8874, 0.9708, 0.2834),    "crop_0.9":      (0.0000, 0.2318, 0.0000),    "crop_0.7":      (0.0000, 0.0010, 0.0000),    "crop_0.5":      (0.0000, 0.0000, 0.0000),    "lumin_0.8":     (0.9992, 0.9970, 0.8196),    "lumin_1.2":     (0.9368, 0.9912, 0.7458),    "contraste_0.8": (0.9990, 0.9962, 0.8096),    "contraste_1.2": (0.9918, 0.9948, 0.8114),}tableau("ROBUSTESSE — taux de verdicts AUTHENTIC sur 5000 images",        ["condition", "base 64 b", "phase B", "96 bits", "gain phase B"],        [[k, f"{v[0]:.1%}", f"{v[1]:.1%}", f"{v[2]:.1%}", f"{v[1]-v[0]:+.1%}"]         for k, v in ROBUSTESSE.items()])

### Trois enseignements**La phase B répare ce qu'on croyait structurel.** Le flou k7 passe de 72,5 % à98,2 %. Ce n'était pas une limite du système mais de *cet entraînement-là*.Le mémoire écrivait qu'« un recadrage à 90 % de la surface annule le canal » —avec la phase B, 23,2 % des images restent identifiables. La phrase est àcorriger.**Le canal à 96 bits est défaillant.** 85,3 % de détection *sans aucuneattaque*, contre 99,9 % à 64 bits. L'anomalie, jusque-là soupçonnée, estconfirmée à grande échelle : ce checkpoint n'est pas utilisable.**La falaise géométrique demeure.** Au-delà de 90 % de surface conservée,aucun modèle ne récupère quoi que ce soit. C'est la limite la plus dure dusystème, et elle est honnêtement rapportée dans le mémoire.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 3.8))noms = [k for k in ROBUSTESSE if not k.startswith("crop_0.5") and not k.startswith("crop_0.7")]x = np.arange(len(noms)); w = 0.27for i, (lab, coul) in enumerate([("base 64 bits", "#2980b9"),                                 ("phase B robuste", "#27ae60"),                                 ("96 bits", "#c0392b")]):    ax.bar(x + (i-1)*w, [ROBUSTESSE[n][i]*100 for n in noms], w, label=lab, color=coul, alpha=0.9)ax.set_xticks(x); ax.set_xticklabels(noms, rotation=45, ha="right", fontsize=8)ax.set_ylabel("verdicts AUTHENTIC (%)"); ax.set_ylim(0, 105)ax.set_title("Robustesse sur 5000 images — trois modèles")ax.legend(fontsize=8, ncol=3, loc="lower left")ax.annotate("la phase B\nrépare le flou k7", xy=(7, 98), xytext=(8.4, 55),            fontsize=8, arrowprops=dict(arrowstyle="->", lw=0.8))plt.tight_layout(); plt.show()

---# 4 · Attaques activesL'attaquant connaît l'algorithme et dispose d'images marquées, mais pas desclés. Aucun seuil du vérifieur n'est modifié. Quatre stratégies, **5000 imagesmarquées et 5000 images vierges**.*Source : `attaques-actives-transplantation-rejeu-mixup-collage-5000-images.json`*

In [ ]:
# Deux campagnes indépendantes : machines, corpus et clés différents.ATTAQUES = {  # nom : (n_pod, faux_pod, n_local, faux_local, ber_moyen)    "aller-retour honnête":  (5000, 4985, 2500, 2497, 0.0005),   # ici : RECONNUS    "A — transplantation":   (5000,    5, 2500,    3, 0.0275),    "B — rejeu de nonce":    (5000,    0, 2500,    0, 0.5020),    "C — mixup":             (5000,    0, 2500,    0, 0.4981),    "D — collage vu par A":  (5000,    2, 2500,    1, 0.2694),    "D — collage vu par B":  (5000,    7, 2500,    1, 0.2715),}lignes = []for k, (np_, fp, nl, fl, ber) in ATTAQUES.items():    if k.startswith("aller"):        lignes.append([k, f"{fp}/{np_} ({fp/np_:.1%})", f"{fl}/{nl} ({fl/nl:.1%})", f"{ber:.4f}"])    else:        lignes.append([k, f"{fp} ({fp/np_:.2%})", f"{fl} ({fl/nl:.2%})", f"{ber:.4f}"])tableau("ATTAQUES ACTIVES — faux AUTHENTIC, deux machines indépendantes",        ["scénario", "pod (5000)", "machine locale (2500)", "BER moyen"], lignes)total = sum(v[1] for k, v in ATTAQUES.items() if not k.startswith("aller"))print()print(f"Total : {total} faux AUTHENTIC sur 25 000 vérifications d'attaque.")print("À 20 images, ces quatre attaques donnaient TOUTES zéro.")

### Ce que ce tableau oblige à corrigerLe mémoire affirme aujourd'hui : « aucune des 100 transplantations testéesn'est acceptée » et « aucune attaque n'a produit de faux AUTHENTIC ».**Les deux phrases sont fausses à 5000 images.**Mais le résultat est plus solide que l'affirmation qu'il remplace, pour troisraisons.**Il est reproduit.** 0,10 % sur le pod, **0,12 % sur une seconde machine**avec un autre corpus et d'autres clés. Ce n'est pas un artefactd'échantillonnage.**Son mécanisme est identifié.** La section 2 montrait que 0,1 % des imagesétrangères passent sous le seuil de 69 bits. Deux mesures indépendantes, lemême chiffre : la liaison au contenu n'est pas binaire, elle laisse passer uneimage sur mille.**Le cœur cryptographique, lui, est intact.** Rejeu et mixup : **0 sur 5000chacun, deux fois**, avec un BER à 0,50 — du bruit pur. Ce qui s'écaille estla liaison au contenu, pas la cryptographie.Un dernier chiffre que le petit échantillon cachait : **15 images légitimessur 5000 ne sont pas reconnues** sur un aller-retour parfaitement honnête,sans aucune attaque.

---# 5 · Bruit additif gaussienL'introduction du mémoire annonce des attaques passives « compression,recadrage, **bruit additif** ». Les deux premières étaient couvertes ; latroisième ne l'était pas. Mesurée sur **5000 images**.*Source : `bruit-additif-gaussien-5000-images/`*

In [ ]:
BRUIT = {  # condition : (bit_acc Ω, hash stable, taux AUTHENTIC)    "aucune":     (0.9995, 1.0000, 0.998),    "σ = 0,02":   (0.9995, 0.8126, 0.996),    "σ = 0,05":   (0.9993, 0.1214, 0.975),    "σ = 0,10":   (0.9975, 0.0112, 0.860),}tableau("BRUIT ADDITIF — 5000 images",        ["condition", "bit_acc de Ω", "hash stable", "AUTHENTIC"],        [[k, f"{v[0]:.4f}", f"{v[1]:.1%}", f"{v[2]:.1%}"] for k, v in BRUIT.items()])print()print("Ce tableau sépare nettement les DEUX canaux du système.")print()print("Le tatouage est presque insensible au bruit : bit_acc reste à 0,9975 même")print("à σ = 0,10, une dégradation pourtant bien visible à l'œil.")print()print("Le hash perceptuel, lui, s'effondre : il ne reste stable que sur 1,1 %")print("des images. C'est le même profil que la falaise géométrique —")print("l'extracteur appris tient, le hachage lâche.")print()print("La conséquence est la bonne : la détection ne tombe qu'à 86 %, en FAUX")print("NÉGATIFS uniquement. Le système refuse de reconnaître, il n'accuse")print("jamais à tort. C'est le profil forensique que le mémoire revendique.")

### Un levier identifiéLe diagnostic de binarisation, joué sur 5000 images, teste huit façons depréparer l'image avant le hachage. Il en sort une piste concrète.*Source : `mesures-machine-locale-5000-images-coco/reparer_5000.json`*

In [ ]:
REMEDES = {  # variante : (dérive du marquage, distance à une autre image)    "référence":                (77.69, 125.25),    "flou σ=1":                 (81.45, 125.11),    "flou σ=3":                 (76.94, 122.68),    "sous-échantillonnage 112": (84.27, 125.31),    "sous-échantillonnage 64":  (75.30, 123.54),    "sous-échantillonnage 32":  (60.65, 114.09),    "flou σ=2 puis 112":        (76.99, 124.49),}tableau("PRÉPARATION DE L'IMAGE AVANT HACHAGE — 5000 images",        ["variante", "dérive du marquage ↓", "autre image ↑", "marge"],        [[k, f"{v[0]:.2f}", f"{v[1]:.2f}", f"{v[1]-v[0]:.2f}"]         for k, v in sorted(REMEDES.items(), key=lambda x: x[1][1]-x[1][0], reverse=True)])print()print("Hacher l'image RÉDUITE À 32×32 fait tomber la dérive due au marquage de")print("77,7 à 60,7 bits, alors que la distance entre contenus distincts ne perd")print("que 11 bits. La marge passe de 47,6 à 53,4 bits, soit +12 %.")print()print("C'est directement actionnable : le seuil actuel laisse passer 0,10 % des")print("contenus étrangers, et une marge plus large réduit ce taux. Le changement")print("tient en deux lignes — on hache une image réduite, rien d'autre ne bouge.")

---# 6 · Le décodeur génératif porte Ω lui-mêmeC'est la contribution centrale. Chaque étage du décodeur est modulé par FiLM àpartir de Ω :$$a \;\leftarrow\; a \odot \bigl(1 + \gamma(\Omega)\bigr) + \beta(\Omega)$$Deux garde-fous de conception :- **identité à l'initialisation** — les têtes sont initialisées à zéro, donc  `tanh(0) = 0` et le décodeur pré-entraîné est bit-à-bit intact au pas 0 ;- **modulation bornée** — `γ = γ_max·tanh(·)` et `β = β_max·tanh(·)`, ce qui  rend la dégradation de l'image impossible par construction.Les étages sont **découverts par passe à blanc**, jamais codés en dur. Sur leDC-AE la configuration annonçait 7 largeurs et l'observation en a trouvé 6,dans l'ordre inverse.Évalué sur **5000 images générées**.*Source : `phaseD-decodeur-conditionne-5000-images-generees.json`*

In [ ]:
PHASE_D = {    "n": 5000, "bit_acc_mediane": 1.0, "bit_acc_moyenne": 0.98775,    "p05": 0.9531, "min": 0.8594,    "omega_exact": 0.529, "verdict_rendu": 1.0, "tau": 69,    "hash": {"légitime": (0.0, 1.000), "JPEG q50": (28.0, 0.9992),             "JPEG q30": (36.0, 0.9950), "transplantation": (111.0, 0.0034)},}tableau("PHASE D — décodeur conditionné, 5000 images GÉNÉRÉES",        ["mesure", "valeur"],        [["bit_acc médiane",              f'{PHASE_D["bit_acc_mediane"]:.4f}'],         ["bit_acc moyenne",              f'{PHASE_D["bit_acc_moyenne"]:.4f}'],         ["bit_acc p05 / min",            f'{PHASE_D["p05"]:.4f} / {PHASE_D["min"]:.4f}'],         ["Ω extrait SANS ERREUR",        f'{PHASE_D["omega_exact"]:.1%}'],         ["verdict rendu (p < 1e-6)",     f'{PHASE_D["verdict_rendu"]:.1%}'],         ["transplantation acceptée",     f'{PHASE_D["hash"]["transplantation"][1]:.2%}']])print()tableau(f'  liaison au contenu sur les images générées (τ = {PHASE_D["tau"]}/256)',        ["cas", "médiane", "sous le seuil"],        [[k, f"{v[0]:.0f}", f"{v[1]:.1%}"] for k, v in PHASE_D["hash"].items()])print()print("Le point que la médiane cachait. Elle vaut 1,0000 — parfait. Mais Ω n'est")print("extrait sans la moindre erreur que sur 52,9 % des images : une sur deux")print("porte au moins un bit faux. Le verdict est tout de même rendu à 100 %,")print("parce que la p-valeur binomiale reste écrasante même avec quelques bits")print("d'erreur. C'est exactement pourquoi il fallait mesurer la DISTRIBUTION")print("et pas seulement la médiane : l'avalanche HMAC exige que chaque image")print("dépasse le seuil individuellement.")

---# 7 · Du prompt au verdict — le tatouage naît dans la générationLa phase D démontrait le principe sur un autoencodeur ImageNet, qu'on ne peutpas piloter : rien ne permet de lui demander une image à partir d'un texte.La **phase E** conditionne le décodeur latent de **SANA**, un vrai modèletexte→image. Un seul facteur change par rapport à la phase D : le modèle.*Source : `04b-decodeur-de-sana-conditionne-40000-pas-bit-acc-0.9867/`*

In [ ]:
PHASE_E = [  # (pas, bit_acc de validation, PSNR in-model)    (    0, 0.5051, 23.74), ( 4000, 0.7703, 18.99), ( 5500, 0.8313, 19.48),    ( 9500, 0.8992, 20.09), (15000, 0.9398, 20.59), (20500, 0.9629, 20.80),    (24000, 0.9734, 20.93), (29500, 0.9801, 21.03), (35000, 0.9836, 21.06),    (39000, 0.9867, 21.24), (40000, 0.9855, 21.20),]fig, a1 = plt.subplots(figsize=(7.5, 3.6))p = [x[0] for x in PHASE_E]; b = [x[1] for x in PHASE_E]; q = [x[2] for x in PHASE_E]a1.plot(p, b, "o-", lw=2, color="#27ae60", label="bit_acc de validation")a1.axhline(0.99, ls="--", lw=1.2, color="#c0392b")a1.text(1000, 0.993, "seuil de l'avalanche HMAC (0,99)", color="#c0392b", fontsize=8)a1.axhline(0.9867, ls=":", lw=1.2, color="#27ae60")a1.text(1000, 0.972, "maximum atteint : 0,9867", color="#27ae60", fontsize=8)a1.set_xlabel("pas d'entraînement"); a1.set_ylabel("bit_acc", color="#27ae60")a1.set_ylim(0.48, 1.02); a1.set_title("Phase E — décodeur latent de SANA conditionné sur Ω")a2 = a1.twinx(); a2.grid(False)a2.plot(p, q, "s--", lw=1.2, color="#2980b9", ms=3.5, label="PSNR (dB)")a2.set_ylabel("PSNR in-model (dB)", color="#2980b9")h1, l1 = a1.get_legend_handles_labels(); h2, l2 = a2.get_legend_handles_labels()a1.legend(h1+h2, l1+l2, fontsize=8, loc="center right")plt.tight_layout(); plt.show()print("Progression monotone de 0,5051 à 0,9867, sans un seul palier.")print("Le PSNR chute à 18,99 quand le conditionneur commence à moduler,")print("puis REMONTE à 21,24 : le décodeur apprend à porter Ω *et* à récupérer")print("sa fidélité. Il n'échange pas l'un contre l'autre.")print()print("À comparer à la phase D, qui atteint 0,989 aux mêmes 40 000 pas sur un")print("autoencodeur ImageNet. Les deux arrivent au même endroit : le")print("conditionnement FiLM se comporte identiquement sur les deux décodeurs.")

### La chaîne complète, exécutée`scripts/ciphermark/chaine_prompt.py` ferme la boucle : une phrase, SANA rendl'image, Ω y est gravé **pendant le décodage**, le vérifieur rend un verdict.Deux modes, et le script refuse de les confondre :- **`posthoc`** — SANA génère, *puis* CipherMark tatoue. Le tatouage reste une  étape ajoutée après coup, que celui qui contrôle le modèle peut sauter.- **`inmodel`** — le décodeur latent est conditionné. Le tatouage naît dans la  génération, il n'y a pas d'étape à sauter.Faute de checkpoint de phase E, le mode `inmodel` s'arrête avec un messageexplicite au lieu de retomber en silence sur `posthoc`.*Source : `demonstration-prompt-vers-verdict-in-model/verdicts-et-p-valeurs.json`*

In [ ]:
INMODEL = [  # (prompt, verdict, erreurs Ω/64, distance hash/69, p-valeur)    ("un phare dans la tempête, photographie", "authentic", 0, 33, 5.42e-20),    ("un renard roux dans la neige",           "no_wm",     1, 77, 3.52e-18),    ("une rue de Yaoundé sous la pluie",       "authentic", 3, 48, 2.37e-15),    ("un atelier d'horloger, gros plan",       "authentic", 4, 63, 3.68e-14),    ("un lac de montagne calme",               "authentic", 0, 43, 5.42e-20),    ("des montgolfières en Cappadoce",         "authentic", 0, 45, 5.42e-20),]tableau("CHAÎNE prompt → verdict, mode IN-MODEL (SANA + décodeur conditionné)",        ["prompt", "verdict", "Ω (err/64)", "hash (/69)", "p-valeur"],        [[p[:38], v, f"{o}", f"{h}", f"{pv:.2e}"] for p, v, o, h, pv in INMODEL])auth = sum(1 for x in INMODEL if x[1] == "authentic")print()print(f"{auth}/{len(INMODEL)} AUTHENTIC. Génération en 1,2 à 3,2 s par image,")print("tatouage compris.")print()print("En mode post-hoc, sur 20 prompts : 20/20.")print()print("L'échec est instructif. Le renard sort no_wm — mais regardez OÙ : Ω est")print("extrait à 1 bit près sur 64, avec p = 3,52e-18. Le témoin est")print("parfaitement lisible. C'est le HASH qui dérive à 77 pour un seuil de 69.")print()print("C'est le mécanisme isolé à la section 2 : le seuil de liaison au contenu,")print("calibré sur des photos naturelles, est TANGENT sur des images générées.")print("L'atelier d'horloger passe de justesse à 63/69, six bits de marge.")print("Le mode in-model n'aggrave pas le problème, il le révèle.")

---# 8 · Les deux limites, mesuréesDeux entraînements ont été menés pour éprouver les frontières de la méthode.Aucun n'atteint le seuil exploitable, et c'est le résultat.

In [ ]:
PHASE_F = [(0, 0.5051), (1000, 0.5344), (2500, 0.5352), (4000, 0.5391),           (5500, 0.5648), (8000, 0.5750), (12000, 0.5898), (20000, 0.5938)]PHASE_G = [(0, 0.5017), (20, 0.5279), (40, 0.5694), (60, 0.5754),           (80, 0.5691), (100, 0.5846), (140, 0.6015), (185, 0.6244), (242, 0.6296)]fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.3))a1.plot([x[0] for x in PHASE_F], [x[1] for x in PHASE_F], "o-", lw=1.8, color="#8e44ad")a1.axhline(0.5, ls=":", color="gray", lw=1); a1.text(500, 0.505, "hasard", fontsize=8, color="gray")a1.set_xlabel("pas"); a1.set_ylabel("bit_acc"); a1.set_ylim(0.48, 1.0)a1.set_title("Phase F — décodeur MaskGIT-VQGAN (autorégressif)")a2.plot([x[0] for x in PHASE_G], [x[1] for x in PHASE_G], "o-", lw=1.8, color="#d35400")a2.axhline(0.5, ls=":", color="gray", lw=1)a2.set_xlabel("époque"); a2.set_ylabel("bit_acc"); a2.set_ylim(0.48, 1.0)a2.set_title("Phase G — injection dans l'espace latent")plt.tight_layout(); plt.show()print("PHASE F — l'agnosticisme architectural a un coût mesurable.")print("  Le conditionneur ne trouve que 3 étages modulables dans MaskGIT")print("  ([512, 128, 3]) contre 6 dans le DC-AE — et l'un des trois est la")print("  sortie RGB, où moduler revient à teinter l'image. Il reste deux vrais")print("  points de modulation, et 0,48 M de paramètres contre 1,92 M.")print("  Maximum 0,5938 après 20 000 pas : elle apprend, mais très lentement.")print()print("PHASE G — la géométrie du latent est le facteur limitant.")print("  À 256 px le latent fait 32x8x8 = 2048 valeurs pour 64 bits, soit")print("  1 PIXEL PAR BIT contre 1024 en pixels. La phase A avait établi que ce")print("  ratio gouverne le plafond ; ce run l'emmène dans la zone la plus")print("  défavorable jamais testée. Maximum 0,6296 à l'époque 242.")print()print("  Point important : la perte descend continûment sous ln 2 = 0,6931.")print("  Ce n'est PAS le point stationnaire trivial qui bloquait les runs à")print("  256 bits. Le réseau apprend quelque chose — il n'en apprend pas assez.")

---# 9 · Bilan## Les trois lacunes que le mémoire prétend combler**1 · Sécurité cryptographique formelle — comblée, au-delà du promis.**5 millions de tirages par largeur, six largeurs, huit propriétés. Zérocollision à partir de 64 bits. Rejeu et mixup à **0 sur 5000, deux fois**,avec un BER à 0,50 — du bruit pur.**2 · Traçabilité individuelle par génération — comblée, c'est l'apport leplus net.** 5000 identités dérivées toutes distinctes, zéro acceptationcroisée. 5000 images générées portant chacune son propre témoin, verdict renduà 100 %. Et la phase E porte ça dans un vrai générateur texte→image, ce quele mémoire n'annonçait pas.**3 · Universalité architecturale — partiellement.** L'introduction exclutexplicitement l'évaluation des autorégressifs. La phase F, menée horspérimètre, plafonne à 0,5938 — et en donne la raison : deux étagesconditionnables au lieu de six.## Ce que la campagne oblige à corriger dans le mémoire| affirmation actuelle | mesure à 5000 images ||---|---|| « aucune des 100 transplantations n'est acceptée » | **0,10 %** acceptées, reproduit à 0,12 % || « aucune attaque n'a produit de faux AUTHENTIC » | **14 sur 25 000** || « un recadrage à 90 % annule le canal » | **faux avec la phase B** : 23,2 % survivent || *(absent du texte)* | **0,3 % de faux négatifs** sur aller-retour honnête |Ces corrections ne fragilisent pas le travail — elles le rendent défendable.« Aucune transplantation acceptée sur 100 essais » tombe dès qu'on demande« et sur 5000 ? ». « 0,10 %, reproduit sur deux machines, avec le seuil quil'explique » ne tombe pas.## Ce qui reste ouvert- **La falaise géométrique** — au-delà de 90 % de surface conservée, aucun  modèle ne récupère rien. C'est la limite la plus dure.- **Le seuil de liaison au contenu** — tangent sur les images générées ; le  sous-échantillonnage à 32×32 élargit la marge de 12 %, piste non exploitée.- **Les modèles autorégressifs** — la phase F dit *pourquoi* c'est difficile,  elle ne dit pas que c'est impossible.- **Le fine-tuning LoRA adversarial** — 30 pas exécutés sur les 2500 prévus.

---# Annexe · Accéder aux données brutesToutes les mesures citées viennent de fichiers déposés sur Google Drive, dans`ciphermark/`, organisé dans l'ordre du pipeline :```01-tatoueur-modeles-de-reference-64-et-128-bits/02-tatoueur-96-bits-canal-plus-large/03-tatoueur-robuste-aux-attaques-passives/04-decodeur-generatif-conditionne-sur-omega-modele-retenu/04b-decodeur-de-sana-conditionne-40000-pas-bit-acc-0.9867/07-validation-a-grande-echelle-5000-images/08-tentatives-a-256-bits-mur-de-capacite/```Chaque dossier porte un `LISEZ-MOI.md`. La cellule suivante monte le Drive etrelit les JSON, pour qui veut vérifier plutôt que croire.

In [ ]:
# Optionnel : relire les mesures depuis le Drive au lieu des valeurs embarquées.try:    from google.colab import drive    drive.mount("/content/drive")    import glob, json, os    RACINE = "/content/drive/MyDrive/ciphermark/07-validation-a-grande-echelle-5000-images"    fichiers = sorted(glob.glob(os.path.join(RACINE, "**", "*.json"), recursive=True))    print(f"{len(fichiers)} fichiers de mesures trouvés :\n")    for f in fichiers:        print("  ", os.path.relpath(f, RACINE))except ImportError:    print("Hors Colab : les valeurs de ce carnet sont embarquées, il s'exécute")    print("tel quel. Le montage du Drive ne sert qu'à recouper les chiffres.")except Exception as e:    print("Drive non monté :", e)